In [ ]:
import time
import json
import pandas as pd
import re
import joblib
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

STOPWORDS = set(stopwords.words('english'))

## Load and prepare data

In [2]:
df = pd.read_csv('../data/LabeledText.csv', encoding='latin-1')
df = df[['Caption', 'LABEL']].copy()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]
    return ' '.join(tokens)

df['clean'] = df['Caption'].apply(clean_text)

X_train, X_test, y_train, y_test = train_test_split(
    df['clean'], df['LABEL'],
    test_size=0.2,
    random_state=42,
    stratify=df['LABEL']
)

## Train Naive Bayes

In [ ]:
t0 = time.time()
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', MultinomialNB())
])
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
runtime = time.time() - t0

print(f'Accuracy: {accuracy_score(y_test, y_pred):.3f}')
print(f'Runtime:  {runtime:.2f}s')
print()
print(classification_report(y_test, y_pred))

report = classification_report(y_test, y_pred, output_dict=True)
meta = {
    'model': 'naive_bayes',
    'accuracy': report['accuracy'],
    'macro_f1': report['macro avg']['f1-score'],
    'negative_f1': report['negative']['f1-score'],
    'neutral_f1': report['neutral']['f1-score'],
    'positive_f1': report['positive']['f1-score'],
    'runtime_seconds': runtime
}
joblib.dump(pipeline, '../models/text/pkl/naive_bayes.pkl')
with open('../models/text/json/naive_bayes_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)